# 02. Exploración inicial de datos

- **Autor:** Jhonner Acosta
- **Fecha:** 2026-08-20
- **Descripción:** Exploración general del dataset de enfermedades cardíacas: descripción de datos, unificación de nulos, conversión de tipos y almacenamiento en parquet.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../../data")

## 1. Carga de datos

In [2]:
df = pd.read_csv(DATA_DIR / "01_raw/corazon.csv")
df.sample(5)

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
2212,62,Female,asymptomatic,160,164,0.0,left ventricular hypertrophy,145,0,6.2,3,3.0,reversable,1
752,57,Male,asymptomatic,165,289,1.0,left ventricular hypertrophy,124,0,1.0,2,3.0,reversable,1
19,49,Male,nontypical,130,266,0.0,normal,171,0,0.6,1,0.0,normal,0
807,64,Female,asymptomatic,180,325,0.0,normal,154,1,0.0,1,0.0,normal,0
2712,56,Male,nontypical,130,221,0.0,left ventricular hypertrophy,163,0,0.0,1,0.0,reversable,0


## 2. Descripción general de los datos

In [3]:
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.info()

Filas: 3030 | Columnas: 14
<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


In [4]:
df.describe(include="all")

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
count,3000,2969,2947,2949,2945,2933.000000,2837,2859,2879,2880,2879,2868,2904,2924
unique,43,5,7,52,154,NaN,8,92,4,42,4,5,7,7
top,58,Male,asymptomatic,120,204,NaN,normal,162,0,0.0,1,0.0,normal,0
freq,187,2017,1394,361,59,NaN,1409,103,1929,944,1353,1684,1599,1576
mean,NaN,NaN,NaN,NaN,NaN,0.148312,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.355470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Conteo de valores nulos por columna
df.isnull().sum()

age            30
sex            61
chest_pain     83
rest_bp        81
chol           85
fbs            97
rest_ecg      193
max_hr        171
exang         151
old_peak      150
slope         151
ca            162
thal          126
disease       106
dtype: int64

# Clasificación de variables del dataset


In [6]:
clasificacion = pd.DataFrame(
    {
        "columna": [
            "age",
            "sex",
            "chest_pain",
            "rest_bp",
            "chol",
            "fbs",
            "rest_ecg",
            "max_hr",
            "exang",
            "old_peak",
            "slope",
            "ca",
            "thal",
            "disease",
        ],
        "tipo_original": df.dtypes.values.astype(str),
        "tipo_correcto": [
            "int8",
            "category nominal",
            "category nominal",
            "int16",
            "int16",
            "bool",
            "category nominal",
            "int16",
            "bool",
            "float32",
            "category ordinal",
            "Int8 nullable",
            "category nominal",
            "bool",
        ],
        "descripcion": [
            "Edad del paciente",
            "Sexo: Male / Female",
            "Tipo de dolor: typical, nontypical, nonanginal, asymptomatic",
            "Presión arterial en reposo (mm Hg)",
            "Colesterol sérico (mg/dl)",
            "Azúcar en ayunas > 120 mg/dl",
            "Resultados del ECG en reposo",
            "Frecuencia cardíaca máxima alcanzada",
            "Angina inducida por ejercicio",
            "Depresión del ST inducida por ejercicio",
            "Pendiente del ST: 1=ascendente, 2=plana, 3=descendente",
            "Número de vasos coloreados por fluoroscopía (0-3)",
            "Prueba de estrés: normal, fixed, reversable",
            "Diagnóstico de enfermedad cardíaca (target)",
        ],
    }
)

clasificacion

,columna,tipo_original,tipo_correcto,descripcion
0,age,str,int8,Edad del paciente
1,sex,str,category nominal,Sexo: Male / Female
2,chest_pain,str,category nominal,"Tipo de dolor: typical, nontypical, nonanginal..."
3,rest_bp,str,int16,Presión arterial en reposo (mm Hg)
4,chol,str,int16,Colesterol sérico (mg/dl)
5,fbs,float64,bool,Azúcar en ayunas > 120 mg/dl
6,rest_ecg,str,category nominal,Resultados del ECG en reposo
7,max_hr,str,int16,Frecuencia cardíaca máxima alcanzada
8,exang,str,bool,Angina inducida por ejercicio
9,old_peak,str,float32,Depresión del ST inducida por ejercicio


## 3. Unificación de valores nulos

Se verifica si existen representaciones no estándar de valores nulos (como `"?"`, cadenas vacías o espacios) y se reemplazan por `np.nan`.

In [7]:
df.replace("?", np.nan, inplace=True)
df.replace("", np.nan, inplace=True)
str_cols = df.select_dtypes(include=["object", "string"]).columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())
df.isnull().sum()

age            30
sex            61
chest_pain     83
rest_bp        81
chol           85
fbs            97
rest_ecg      193
max_hr        171
exang         151
old_peak      150
slope         151
ca            162
thal          126
disease       106
dtype: int64

In [8]:
# Eliminar filas con valores inválidos en columnas categóricas
valores_validos = {
    "sex": ["Male", "Female"],
    "chest_pain": ["typical", "nontypical", "nonanginal", "asymptomatic"],
    "rest_ecg": ["normal", "left ventricular hypertrophy", "ST-T wave abnormality"],
    "thal": ["normal", "fixed", "reversable"],
}
for col, validos in valores_validos.items():
    mask = ~df[col].isin(validos) & df[col].notna()
    print(f"{col}: {mask.sum()} filas corruptas eliminadas")
    df = df[~mask]

# Limpiar slope: convertir a numérico primero, luego filtrar
df["slope"] = pd.to_numeric(df["slope"], errors="coerce")
# Eliminar filas donde slope es NaN (valores no convertibles como "afd") o fuera de rango
mask_invalido = ~df["slope"].isin([1.0, 2.0, 3.0])
print(f"slope: {mask_invalido.sum()} filas con valores inválidos o NaN eliminadas")
df = df[~mask_invalido].reset_index(drop=True)
print(f"Filas restantes: {len(df)}")

sex: 3 filas corruptas eliminadas
chest_pain: 3 filas corruptas eliminadas
rest_ecg: 4 filas corruptas eliminadas
thal: 4 filas corruptas eliminadas
slope: 152 filas con valores inválidos o NaN eliminadas
Filas restantes: 2864


In [9]:
cols_numericas = ["age", "rest_bp", "chol", "max_hr", "old_peak", "ca"]
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")
# ... resto de conversiones

## 4. Conversión de tipos de datos

### 4.1 Variables numéricas

In [10]:
# Convertir columnas numéricas de string a número primero (por si quedaron como object)
cols_numericas = ["age", "rest_bp", "chol", "max_hr", "old_peak", "ca"]
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("NaN por columna numérica:")
print(df[cols_numericas].isnull().sum())

# Rellenar NaN con mediana y convertir tipos
df["age"] = df["age"].fillna(df["age"].median()).astype("int8")
df["rest_bp"] = df["rest_bp"].fillna(df["rest_bp"].median()).astype("int16")
df["chol"] = df["chol"].fillna(df["chol"].median()).astype("int16")
df["max_hr"] = df["max_hr"].fillna(df["max_hr"].median()).astype("int16")
df["old_peak"] = df["old_peak"].fillna(df["old_peak"].median()).astype("float32")
df["ca"] = df["ca"].astype("Int8")

print("Tipos numéricos asignados correctamente")

NaN por columna numérica:
age         11
rest_bp     47
chol        51
max_hr      25
old_peak     2
ca          40
dtype: int64
Tipos numéricos asignados correctamente


### 4.2 Variables booleanas

In [11]:
df["fbs"] = df["fbs"].map({0: False, 1: True}).astype("boolean")
df["exang"] = df["exang"].map({0: False, 1: True}).astype("boolean")
df["disease"] = (pd.to_numeric(df["disease"], errors="coerce") > 0).astype("boolean")

print("Variables booleanas asignadas:")
print(df[["fbs", "exang", "disease"]].dtypes)
print("\nNulos en booleanas:")
print(df[["fbs", "exang", "disease"]].isnull().sum())

Variables booleanas asignadas:
fbs        boolean
exang      boolean
disease    boolean
dtype: object

Nulos en booleanas:
fbs          46
exang      2864
disease       0
dtype: int64


### 4.3 Variables categóricas nominales

In [12]:
def filtrar_categoricos_invalidos(df, valores_validos):
    for col, validos in valores_validos.items():
        mask = ~df[col].isin(validos) & df[col].notna()
        print(f"{col}: {mask.sum()} filas corruptas eliminadas")
        df = df[~mask]
    return df.reset_index(drop=True)


valores_validos = {
    "sex": ["Male", "Female"],
    "chest_pain": ["typical", "nontypical", "nonanginal", "asymptomatic"],
    "rest_ecg": ["normal", "left ventricular hypertrophy", "ST-T wave abnormality"],
    "thal": ["normal", "fixed", "reversable"],
}

df = filtrar_categoricos_invalidos(df, valores_validos)

# Limpiar slope
df["slope"] = pd.to_numeric(df["slope"], errors="coerce")
mask = ~df["slope"].isin([1.0, 2.0, 3.0]) & df["slope"].notna()
print(f"slope: {mask.sum()} filas corruptas eliminadas")
df = df[~mask].reset_index(drop=True)

print(f"\nFilas restantes: {len(df)}")

sex: 0 filas corruptas eliminadas
chest_pain: 0 filas corruptas eliminadas
rest_ecg: 0 filas corruptas eliminadas
thal: 0 filas corruptas eliminadas
slope: 0 filas corruptas eliminadas

Filas restantes: 2864


### 4.4 Variables categóricas ordinales

In [13]:
# slope: 1=ascendente, 2=plana, 3=descendente
slope_order = pd.CategoricalDtype(categories=[1, 2, 3], ordered=True)
df["slope"] = df["slope"].astype(slope_order)

print("slope ordenado:", df["slope"].cat.categories.tolist())

slope ordenado: [1, 2, 3]


### 4.5 Verificación del schema final

In [14]:
df_raw = pd.read_csv(DATA_DIR / "01_raw/corazon.csv")
print(f"Filas en CSV: {len(df_raw)}")
print(df_raw["slope"].value_counts())
print(df_raw["slope"].dtype)

Filas en CSV: 3030
slope
1      1353
2      1323
3       202
afd       1
Name: count, dtype: int64
str


In [15]:
df.info()
print(f"\nMemoria utilizada: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

<class 'pandas.DataFrame'>
RangeIndex: 2864 entries, 0 to 2863
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2864 non-null   int8    
 1   sex         2818 non-null   str     
 2   chest_pain  2817 non-null   str     
 3   rest_bp     2864 non-null   int16   
 4   chol        2864 non-null   int16   
 5   fbs         2818 non-null   boolean 
 6   rest_ecg    2818 non-null   str     
 7   max_hr      2864 non-null   int16   
 8   exang       0 non-null      boolean 
 9   old_peak    2864 non-null   float32 
 10  slope       2864 non-null   category
 11  ca          2824 non-null   Int8    
 12  thal        2842 non-null   str     
 13  disease     2864 non-null   boolean 
dtypes: Int8(1), boolean(3), category(1), float32(1), int16(3), int8(1), str(4)
memory usage: 256.7 KB

Memoria utilizada: 256.7 KB


## 5. Almacenamiento en formato Parquet

In [16]:
output_path = DATA_DIR / "02_intermediate/corazon_type_fixed.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_path, index=False)

print(f"Dataset guardado en: {output_path}")
print(f"Tamaño del archivo: {output_path.stat().st_size / 1024:.1f} KB")

Dataset guardado en: ../../data/02_intermediate/corazon_type_fixed.parquet
Tamaño del archivo: 18.8 KB


### Verificación del archivo guardado

In [17]:
df_check = pd.read_parquet(output_path)
print(f"Filas: {df_check.shape[0]} | Columnas: {df_check.shape[1]}")
df_check.info()

Filas: 2864 | Columnas: 14
<class 'pandas.DataFrame'>
RangeIndex: 2864 entries, 0 to 2863
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         2864 non-null   int8   
 1   sex         2818 non-null   str    
 2   chest_pain  2817 non-null   str    
 3   rest_bp     2864 non-null   int16  
 4   chol        2864 non-null   int16  
 5   fbs         2818 non-null   boolean
 6   rest_ecg    2818 non-null   str    
 7   max_hr      2864 non-null   int16  
 8   exang       0 non-null      boolean
 9   old_peak    2864 non-null   float32
 10  slope       2864 non-null   int64  
 11  ca          2824 non-null   Int8   
 12  thal        2842 non-null   str    
 13  disease     2864 non-null   boolean
dtypes: Int8(1), boolean(3), float32(1), int16(3), int64(1), int8(1), str(4)
memory usage: 276.3 KB


In [18]:
# Debug command retained as comment to keep this cell valid Python for pre-commit\n# grep -n "valores_validos\|filtrar_categoricos" notebooks/2-exploration/02.Exploracion_datos-jjac-2026-08-20.ipynb\n